# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published Date: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

List all the RecordSets in this dataset, showing their `@id`, `name`, and the available field `@id`s within each RecordSet.

In [ ]:
# List available record sets and fields using @id references
record_sets = list(dataset.record_sets)
print(f"Number of RecordSets found: {len(record_sets)}\n")
record_sets_ids = []
for record_set in record_sets:
    print(f"@id: {record_set.id}")
    print(f"  Name: {getattr(record_set, 'name', 'N/A')}")
    print(f"  Description: {getattr(record_set, 'description', 'N/A')}")
    print("  Fields:")
    for field in getattr(record_set, 'fields', []):
        print(f"    - @id: {field.id}, name: {getattr(field, 'name', 'N/A')}")
    print('-' * 40)
    record_sets_ids.append(record_set.id)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set by @id
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet @id '{record_set_id}'. Columns:")
        print(df.columns.tolist())
        print('-'*40)
    else:
        print(f"No records loaded for RecordSet @id '{record_set_id}'")

# For demonstration, pick the first record set with data
main_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        main_record_set_id = k
        break
if main_record_set_id:
    print(f"First RecordSet with records: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print('No dataframes with data were found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Attempt to select a numeric field for EDA
import numpy as np

df = dataframes[main_record_set_id]
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric fields found: {numeric_fields}")

if numeric_fields:
    numeric_field = numeric_fields[0]
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by another field (if present)
    group_candidates = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
    else:
        group_field = None
        print("No suitable object-type field found for grouping.")
else:
    numeric_field = None
    print('No numeric fields found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print('No numeric field for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**In this notebook, we have loaded a FAIR-compliant mass spectrometry dataset of extracellular vesicles from human mast cells using the `mlcroissant` library. We explored record sets and fields, loaded tabular data, performed simple EDA, and visualized relevant numeric columns. For more advanced analysis, consult the specific protocol notes or documentation available via the dataset schema.**